# 🎨 使用 GitHub Models 的代理设计模式 (Python)

## 📋 学习目标

本笔记本演示了使用 Microsoft Agent Framework 与 GitHub Models 集成构建智能代理的基本设计模式。您将学习经过验证的模式和架构方法，使代理更加健壮、可维护和有效。

**涵盖的核心设计模式：**
- 🏗️ **代理工厂模式**：标准化的代理创建和配置
- 🔧 **工具注册表模式**：管理代理功能的有组织方法
- 🧵 **对话管理**：多轮交互的有效模式
- 🔄 **响应处理**：处理代理输出的最佳实践

## 🎯 关键架构概念

### 设计原则
- **关注点分离**：代理逻辑、工具和配置之间的清晰边界
- **可组合性**：从可重用组件构建复杂代理
- **可扩展性**：允许轻松添加新功能的模式
- **可测试性**：便于单元测试和验证的设计

### GitHub Models 集成
- **API 兼容性**：利用 OpenAI 兼容的端点
- **模型选择**：为不同用例选择合适的模型
- **速率限制**：优雅处理 API 约束
- **错误恢复**：健壮的错误处理和重试模式

## 🔧 技术架构

### 核心组件
- **Microsoft Agent Framework**：支持 GitHub Models 的 Python 实现
- **GitHub Models API**：访问最先进的语言模型
- **OpenAI 客户端模式**：标准化的 API 交互模式
- **环境配置**：安全灵活的配置管理

### 设计模式优势
- **可维护性**：清晰的代码组织和结构
- **可扩展性**：随应用需求增长的模式
- **可靠性**：处理边缘情况的成熟方法
- **性能**：高效的资源利用和 API 使用

## ⚙️ 先决条件和设置

**所需依赖：**
```bash

pip install agent-framework-core  -U

```

**环境配置 (.env 文件)：**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

**GitHub Models 访问：**
- 具有 Models 访问权限的 GitHub 账户
- 具有适当权限的个人访问令牌
- 了解速率限制和使用模式

## 📚 设计模式类别

### 1. **创建模式**
- 代理工厂和构建器模式
- 配置管理模式
- 代理服务的依赖注入

### 2. **行为模式**
- 工具执行和编排
- 对话流管理  
- 响应处理和格式化

### 3. **集成模式**
- GitHub Models API 集成
- 错误处理和重试逻辑
- 资源管理和清理

## 🚀 展示的最佳实践

- **清洁架构**：分层设计，职责明确
- **错误处理**：全面的异常管理
- **配置**：基于环境的不同环境设置
- **测试**：启用有效单元测试和集成测试的模式
- **文档**：具有明确意图的自文档化代码

准备好探索专业的代理设计模式了吗？让我们构建一些健壮的东西！ 🌟

In [ ]:
# ! pip install agent-framework-core  -U

In [2]:
# 📦 导入代理设计模式的核心库
import os                     # 用于配置管理的环境变量访问
from random import randint    # 用于工具功能的随机选择实用程序

from dotenv import load_dotenv  # 安全的环境配置加载

In [3]:
# 🤖 导入 Microsoft Agent Framework 组件  
# ChatAgent: 遵循工厂模式的核心代理编排类
# OpenAIChatClient: 遵循适配器模式的 GitHub Models 集成
from agent_framework import ChatAgent
from agent_framework.openai import OpenAIChatClient

In [4]:
# 🔧 配置加载模式
# 实现安全凭证处理的配置管理模式
# 这遵循云原生应用程序的外部配置原则
load_dotenv()

True

In [5]:
# 🛠️ 工具函数设计模式
# 为可插拔代理功能实现策略模式
# 这展示了业务逻辑与代理编排的清晰分离
def get_random_destination() -> str:
    """使用存储库模式获取随机度假目的地。
    
    此函数展示了几种设计模式：
    - 策略模式：目的地选择的可互换算法
    - 存储库模式：封装数据访问逻辑
    - 工厂方法：按需创建目的地对象
    
    返回：
        str: 遵循一致格式的随机选择目的地
    """
    # 数据存储库模式：集中式目的地数据管理
    destinations = [
        "Barcelona, Spain",      # 地中海文化中心
        "Paris, France",         # 欧洲艺术中心
        "Berlin, Germany",       # 欧洲历史首都
        "Tokyo, Japan",          # 亚洲科技大都市
        "Sydney, Australia",     # 大洋洲沿海城市
        "New York, USA",         # 美国城市中心
        "Cairo, Egypt",          # 非洲历史首都
        "Cape Town, South Africa", # 非洲风景目的地
        "Rio de Janeiro, Brazil",  # 南美海滩城市
        "Bali, Indonesia"          # 东南亚岛屿天堂
    ]
    
    # 工厂方法模式：按需创建目的地选择
    return destinations[randint(0, len(destinations) - 1)]

In [6]:
openai_chat_client = OpenAIChatClient(base_url=os.environ.get("API_URL"), api_key=os.environ.get("API_KEY"), model_id=os.environ.get("MODEL_FREE_8B"))

In [7]:
AGENT_NAME ="TravelAgent"

AGENT_INSTRUCTIONS = """你是一个有用的 AI 代理，可以帮助客户计划假期。

重要：当用户指定目的地时，始终为该位置规划。仅当用户未指定偏好时才建议随机目的地。

对话开始时，用以下消息介绍自己：
"你好！我是你的 TravelAgent 助手。我可以帮你计划假期并为你推荐有趣的目的地。你可以问我以下事情：
1. 计划前往特定地点的一日游
2. 推荐随机度假目的地
3. 寻找具有特定特色的目的地（海滩、山脉、历史遗迹等）
4. 如果你不喜欢我的第一个建议，计划替代行程

今天你想让我帮你计划什么样的旅行？"

始终优先考虑用户偏好。如果他们提到特定目的地，如"巴厘岛"或"巴黎"，请专注于该位置的规划，而不是建议替代方案。
"""

In [8]:
agent = ChatAgent(
        name = AGENT_NAME,
        chat_client=openai_chat_client,
        instructions=AGENT_INSTRUCTIONS,
        tools=[get_random_destination]
)

In [9]:
thread = agent.get_new_thread()

In [10]:
response1 = await agent.run("Plan me a day trip",thread= thread)

In [11]:

last_message = response1.messages[-1]
text_content = last_message.contents[0].text
print("Travel plan:")
print(text_content)

Travel plan:
好的！我为你规划了一日游行程，目的地是悉尼，澳大利亚。以下是详细的行程安排：

**早上：**
- **上午 9:00 - 11:00**：开始你的悉尼一日游，前往著名的**悉尼歌剧院**。你可以参观其内部，了解这座标志性建筑的历史和建筑特色，还可以参加导览团，感受其独特的设计灵感。

**中午：**
- **中午 12:00 - 1:30**：在**悉尼海港大桥**附近享用午餐，选择当地受欢迎的餐厅，比如靠近海港的**The Vue**或**Barrel**，品尝正宗的澳洲美食，例如肉派、海鲜或烧烤。

**下午：**
- **下午 2:00 - 4:00**：漫步至**邦迪海滩（Bondi Beach）**，这是悉尼最受欢迎的海滩之一。你可以享受阳光、沙滩，或者进行一些轻松的活动，如骑自行车或散步。此外，这里的海景非常壮观，是拍照的好地方。

**傍晚：**
- **下午 5:00 - 6:30**：前往**岩石区（The Rocks）**，这是悉尼最古老的地区之一，充满了殖民时期的建筑和历史氛围。可以在这里购物，品尝当地的小吃，比如新鲜的海鲜和咖啡。

**晚上：**
- **晚上 7:00 - 9:00**：结束一天的行程，前往**悉尼塔龙加动物园（Taronga Zoo）**或**海港大桥观景台**欣赏悉尼的夜景。如果时间允许，可以参观**悉尼的星光大道（Starfield）**，体验独特的灯光艺术。

长途旅行可能需要更详细的计划，但一天的时间足以让你感受到悉尼的魅力。你觉得这样的行程安排怎么样？如果需要调整，请告诉我！


In [12]:
response2 = await agent.run("I don't like that destination. Plan me another vacation.",thread= thread)

In [13]:
last_message = response2.messages[-1]
text_content = last_message.contents[0].text
print("Change plan:")
print(text_content)

Change plan:
好的！我为你重新规划了一个度假行程，这次的目的地是**巴塞罗那，西班牙**。巴塞罗那以其丰富的历史、迷人的建筑和充满活力的文化而闻名，非常适合一天的深度体验。以下是详细的行程安排：

**早上：**
- **上午 9:00 - 11:00**：前往**圣家堂（Sagrada Família）**，这是安东尼·高迪的杰作，也是巴塞罗那最著名的地标之一。你可以参观其内部，感受高迪独特而震撼的艺术风格。

**中午：**
- **中午 12:00 - 1:30**：在**兰布拉大道（La Rambla）**附近的餐厅享用午餐，推荐尝试地道的西班牙美食，比如海鲜饭（Paella）和Tapas小点心。可以选择传统的餐厅如**Casa Pitu**或**La Boqueria市场**附近的餐馆。

**下午：**
- **下午 2:00 - 4:00**：漫步**蒙特惠奇山（Montjuïc Hill）**，沿途可以欣赏到巴塞罗那的壮丽全景，尤其是俯瞰城市的视野非常震撼。此外，山上有**蒙特惠奇城堡（Castell de Montjuïc）**和**巴塞罗那奥运会遗址**，值得一看。

**傍晚：**
- **下午 5:00 - 6:30**：前往**米拉之家（Casa Milà）**，也被称为“石头之家”，这座建筑同样由高迪设计，充满精致的雕塑和独特的建筑细节，非常适合拍照。

**晚上：**
- **晚上 7:00 - 9:00**：结束一天的行程，前往**波盖利亚市场（La Boqueria）**，这里的美食和色彩丰富的摊位非常吸引人。如果你喜欢夜生活，可以去**波盖利亚市场周边的餐厅**或**Las Ramblas小吃街**，体验当地夜生活。

巴塞罗那这一天的行程会充分让你感受到这座城市的魅力。如果你对某个活动感兴趣或想了解更多细节，请随时告诉我！
